In [12]:
# standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import mlflow
import mlflow.xgboost
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# load data
fraud_df = pd.read_csv('../data/creditcard.csv')

# recreate all engineered features
fraud_df['Amount_log'] = np.log1p(fraud_df['Amount'])
fraud_df['Hour'] = (fraud_df['Time'] / 3600 % 24).astype(int)
fraud_df = fraud_df.sort_values('Time').reset_index(drop=True)
fraud_df['rolling_avg_amount'] = fraud_df['Amount'].rolling(window=100, min_periods=1).mean()
fraud_df['rolling_std_amount'] = fraud_df['Amount'].rolling(window=100, min_periods=1).std()
fraud_df['rolling_fraud_density'] = fraud_df['Class'].rolling(window=100, min_periods=1).mean() * 100
fraud_df['rolling_txn_count']=fraud_df['Amount'].rolling(window=100, min_periods=1).count()
fraud_df['rolling_std_amount'] = fraud_df['rolling_std_amount'].fillna(0)  # fill NaN at row 0

print(f"Data loaded: {fraud_df.shape}")
print(f"Fraud rate: {fraud_df['Class'].mean()*100:.3f}%")

Data loaded: (284807, 37)
Fraud rate: 0.173%


In [13]:

features_to_drop = ['Time',                  # replaced by Hour
                    'Amount',                # replaced by Amount_log
                    'Class',                 # target variable
                    'rolling_fraud_density', # IV=0, target leakage
                    'rolling_txn_count',     # IV=0, no predictive power
                    'V15',                   # IV=0.071, too weak
                    'V22',                   # IV=0.085, too weak
                    'V26',                   # IV=0.087, too weak
                    'rolling_avg_amount',    # IV=0.046, too weak
                    ]

X = fraud_df.drop(columns=features_to_drop)
y = fraud_df['Class']

print(f"Features: {X.shape[1]}")
print(f"Feature names: {X.columns.tolist()}")



Features: 28
Feature names: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V23', 'V24', 'V25', 'V27', 'V28', 'Amount_log', 'Hour', 'rolling_std_amount']


In [14]:
train_idx = int (len(fraud_df)*0.70)
validate_idx = int(len(fraud_df)*0.85)

X_train = X.iloc[:train_idx]
X_validate = X.iloc[train_idx:validate_idx]
X_test = X.iloc[validate_idx:]

y_train    = y.iloc[:train_idx]
y_validate = y.iloc[train_idx:validate_idx]
y_test     = y.iloc[validate_idx:]

print(f"Train {X_train.shape[0]:} rows | Fraud rate : {y_train.mean()*100:.3f}%")
print(f"Validate:  {X_validate.shape[0]:}rows | Fraud rate: {y_validate.mean()*100:.3f}%")
print(f"X_test: {X_test.shape[0]:}rows |Fraud rate:{y_test.mean()*100:.3f}%")
print(f"\nTotal: {X_train.shape[0] +X_validate.shape[0]+X_test.shape[0]:,} rows")
print(f"Split: {X_train.shape[0]/len(fraud_df)*100:.0f}% / "
      f"{X_validate.shape[0]/len(fraud_df)*100:.0f}% / "
      f"{X_test.shape[0]/len(fraud_df)*100:.0f}%")

Train 199364 rows | Fraud rate : 0.193%
Validate:  42721rows | Fraud rate: 0.131%
X_test: 42722rows |Fraud rate:0.122%

Total: 284,807 rows
Split: 70% / 15% / 15%


In [15]:
# calculate class weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.1f}")
print(f"This tells XGBoost: missing 1 fraud = same cost as misclassifying {scale_pos_weight:.0f} legitimate")

scale_pos_weight: 518.2
This tells XGBoost: missing 1 fraud = same cost as misclassifying 518 legitimate


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# scale features — required for logistic regression
scaler = StandardScaler()
X_train_scaled    = scaler.fit_transform(X_train)
X_validate_scaled = scaler.transform(X_validate)
X_test_scaled     = scaler.transform(X_test)



print(f"Train scaled shape: {X_train_scaled.shape}")

Train scaled shape: (199364, 28)


In [17]:

lr = LogisticRegression(
    class_weight='balanced',  
    max_iter=1000,            
    C=0.1,                    
                              
    solver='lbfgs',           
    random_state=42
)
lr.fit(X_train_scaled, y_train)
print("Logistic Regression Done ")

# 2. Random Forest

rf = RandomForestClassifier(
    n_estimators=200,        
    max_depth=10,            
    min_samples_split=10,    
    min_samples_leaf=5,       
    max_features='sqrt',      
    class_weight='balanced',  
    random_state=42,
    n_jobs=-1                 
)
rf.fit(X_train, y_train)
print("Random Forest Done ")

# 3. XGBoost 

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight, 
    n_estimators=200,                  
    max_depth=6,                       
    learning_rate=0.1,                
    subsample=0.8,                     
    colsample_bytree=0.8,              
    min_child_weight=5,                
    gamma=1,                           
    random_state=42,
    eval_metric='aucpr',              
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print("XGBoost Done ")

Logistic Regression Done 
Random Forest Done 
XGBoost Done 


In [18]:


lr_probs_val    = lr.predict_proba(X_validate_scaled)[:, 1]
rf_probs_val    = rf.predict_proba(X_validate)[:, 1]
xgb_probs_val   = xgb_model.predict_proba(X_validate)[:, 1]

lr_probs_test   = lr.predict_proba(X_test_scaled)[:, 1]
rf_probs_test   = rf.predict_proba(X_test)[:, 1]
xgb_probs_test  = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions generated ")
print(f"\nSample XGBoost fraud scores (first 10):")
print(xgb_probs_val[:10].round(4))
print(f"\nMax fraud score: {xgb_probs_val.max():.4f}")
print(f"Min fraud score: {xgb_probs_val.min():.4f}")
print(f"Mean fraud score: {xgb_probs_val.mean():.4f}")

Predictions generated 

Sample XGBoost fraud scores (first 10):
[0.     0.     0.0001 0.     0.     0.     0.     0.     0.0001 0.    ]

Max fraud score: 1.0000
Min fraud score: 0.0000
Mean fraud score: 0.0015


In [9]:
lr_probs_val

array([0.0462943 , 0.01155101, 0.04271205, ..., 0.37374223, 0.05294172,
       0.0662683 ], shape=(42721,))